# DemoNotebook — Adversarial & Compression Perturbation Visualiser

Side-by-side image comparisons across five pipeline variants:
**Clean → Compressed → Attacked → Compressed→Attacked → Attacked→Compressed**

Each panel shows predicted class and confidence. Green border = correct prediction, red = wrong.

In [ ]:
# Uncomment on Kaggle / fresh environment
# !pip install -q detectors datasets
# !pip install -q compressai timm

In [ ]:
import os, random, math
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Tuple
from io import BytesIO

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

import timm
from compressai.zoo import cheng2020_attn
from PIL import Image

import matplotlib
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Configuration
Edit the cell below to choose dataset, model, image, compression, and attack.

In [ ]:
# ─── USER CONFIGURATION ───────────────────────────────────────────────────────
DATASET_NAME    = "cifar10"          # "cifar10" | "imagenet"
MODEL_NAME      = "resnet50_cifar10" # see model registry below

IMAGE_INDEX     = 7                  # index into the test set

COMPRESSION_KEY = "jpeg"             # "jpeg" | "jpeg2000" | "pca" | "patchsvd" | "lic_roi"
QUALITY         = 25                 # 25 | 50 | 75

ATTACK_KEY      = "pgd"              # "fgsm" | "pgd" | "apgd"
EPSILON         = 8 / 255            # perturbation budget (L-inf)

# ImageNet local path (only used if DATASET_NAME == "imagenet")
IMAGENET_VAL_PATH = "/data/shared/imagenet/val"

# Output figure path (set to None to skip saving)
FIGURE_SAVE_PATH = f"demo_{DATASET_NAME}_{COMPRESSION_KEY}_q{QUALITY}_{ATTACK_KEY}.png"
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

_NORMS = {
    "cifar10":   ((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    "cifar100":  ((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    "imagenet":  ((0.485,  0.456,  0.406),  (0.229,  0.224,  0.225)),
}

DATA_MEAN, DATA_STD = _NORMS[DATASET_NAME]
NUM_CLASSES = {"cifar10": 10, "cifar100": 100, "imagenet": 1000}[DATASET_NAME]

DATA_MEAN_T = torch.tensor(DATA_MEAN, device=device).view(1, 3, 1, 1)
DATA_STD_T  = torch.tensor(DATA_STD,  device=device).view(1, 3, 1, 1)

def denormalize(x: torch.Tensor) -> torch.Tensor:
    return (x * DATA_STD_T + DATA_MEAN_T).clamp(0.0, 1.0)

def renormalize(x: torch.Tensor) -> torch.Tensor:
    return (x - DATA_MEAN_T) / DATA_STD_T

def project_linf_pixel(x_adv: torch.Tensor, x0: torch.Tensor, eps: float) -> torch.Tensor:
    return torch.max(torch.min(x_adv, x0 + eps), x0 - eps).clamp(0.0, 1.0)

print(f"Dataset: {DATASET_NAME}  |  Classes: {NUM_CLASSES}")

In [ ]:
if DATASET_NAME == "cifar10":
    transform = T.Compose([T.ToTensor(), T.Normalize(DATA_MEAN, DATA_STD)])
    test_set = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
    img_norm, true_label = test_set[IMAGE_INDEX]
    img_norm = img_norm.unsqueeze(0).to(device)
    CLASS_NAMES = CIFAR10_CLASSES

elif DATASET_NAME == "imagenet":
    transform = T.Compose([
        T.Resize(256), T.CenterCrop(224),
        T.ToTensor(), T.Normalize(DATA_MEAN, DATA_STD),
    ])
    test_set = torchvision.datasets.ImageFolder(root=IMAGENET_VAL_PATH, transform=transform)
    img_norm, true_label = test_set[IMAGE_INDEX]
    img_norm = img_norm.unsqueeze(0).to(device)
    CLASS_NAMES = torchvision.models.ResNet50_Weights.DEFAULT.meta["categories"]

true_label_t = torch.tensor([true_label], device=device)
true_class   = CLASS_NAMES[true_label]
print(f"Image #{IMAGE_INDEX}  |  True class: '{true_class}'  (label={true_label})")

In [ ]:
_MODEL_LOADERS: Dict[str, Callable[[], nn.Module]] = {
    "resnet18_cifar10":  lambda: timm.create_model("resnet18_cifar10",  pretrained=True),
    "resnet50_cifar10":  lambda: timm.create_model("resnet50_cifar10",  pretrained=True),
    "resnet18_cifar100": lambda: timm.create_model("resnet18_cifar100", pretrained=True),
    "resnet50_cifar100": lambda: timm.create_model("resnet50_cifar100", pretrained=True),
    "resnet18_imagenet": lambda: torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT),
    "resnet50_imagenet": lambda: torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT),
}

assert MODEL_NAME in _MODEL_LOADERS, f"Unknown model '{MODEL_NAME}'. Options: {list(_MODEL_LOADERS)}"
model = _MODEL_LOADERS[MODEL_NAME]().to(device).eval()
print(f"Model loaded: {MODEL_NAME}")

In [ ]:
class Perturbation(ABC):
    def __init__(self, name: str):
        self.name = name

    @abstractmethod
    def apply(self, model: nn.Module, images: torch.Tensor,
              labels: torch.Tensor, device: torch.device) -> torch.Tensor:
        pass

In [ ]:
class JpegPerturbation(Perturbation):
    def __init__(self, quality: int):
        super().__init__(f"jpeg_q{quality}")
        self.quality = int(quality)
        self._to_pil   = T.ToPILImage()
        self._to_tensor = T.ToTensor()

    def apply(self, model, images, labels, device):
        pixels = denormalize(images.detach().to(device)).cpu()
        out = []
        for img in pixels:
            buf = BytesIO()
            self._to_pil(img).save(buf, format="JPEG", quality=self.quality)
            buf.seek(0)
            out.append(self._to_tensor(Image.open(buf).convert("RGB")))
        return renormalize(torch.stack(out).to(device))


class Jpeg2000Perturbation(Perturbation):
    def __init__(self, quality: float):
        super().__init__(f"jpeg2000_q{int(quality)}")
        self.quality = float(quality)
        self._to_pil    = T.ToPILImage()
        self._to_tensor = T.ToTensor()

    @staticmethod
    def _q_to_rate(q: float) -> float:
        q = float(max(1.0, min(100.0, q)))
        t = (100.0 - q) / 99.0
        return 1.0 * (100.0 ** t)

    def apply(self, model, images, labels, device):
        pixels = denormalize(images.detach().to(device)).cpu()
        rate   = self._q_to_rate(self.quality)
        out    = []
        for img in pixels:
            buf = BytesIO()
            self._to_pil(img).save(buf, format="JPEG2000",
                                   quality_mode="rates", quality_layers=[rate],
                                   irreversible=True)
            buf.seek(0)
            out.append(self._to_tensor(Image.open(buf).convert("RGB")))
        return renormalize(torch.stack(out).clamp(0, 1).to(device))

In [ ]:
class PcaPerturbation(Perturbation):
    def __init__(self, quality: float):
        super().__init__(f"pca_q{int(quality)}")
        self.quality = float(quality)

    def apply(self, model, images, labels, device):
        x = denormalize(images.detach().to(device))
        B, C, H, W = x.shape
        k = max(1, int(round(self.quality / 100.0 * min(H, W))))
        x_flat = x.view(B * C, H, W)
        x_out  = torch.empty_like(x_flat)
        for i in range(x_flat.size(0)):
            A   = x_flat[i]
            mu  = A.mean(dim=1, keepdim=True)
            U, S, Vh = torch.linalg.svd(A - mu, full_matrices=False)
            x_out[i] = (U[:, :k] * S[:k]) @ Vh[:k, :] + mu
        return renormalize(x_out.view(B, C, H, W).clamp(0, 1))


class PatchSVDPerturbation(Perturbation):
    def __init__(self, quality: float, patch_size: int = 8):
        super().__init__(f"patchsvd_q{int(quality)}_p{patch_size}")
        self.quality    = float(quality)
        self.patch_size = int(patch_size)

    def apply(self, model, images, labels, device):
        x = denormalize(images.detach().to(device))
        B, C, H, W = x.shape
        p = self.patch_size
        pad_h = (p - H % p) % p
        pad_w = (p - W % p) % p
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h))
        H2, W2 = x.shape[2], x.shape[3]
        patches = x.unfold(2, p, p).unfold(3, p, p)
        GH, GW  = patches.shape[2], patches.shape[3]
        flat    = patches.contiguous().view(-1, p, p)
        U, S, Vh = torch.linalg.svd(flat, full_matrices=False)
        k = max(1, int(round(self.quality / 100.0 * p)))
        recon = ((U[:, :, :k] * S[:, :k].unsqueeze(1)) @ Vh[:, :k, :])
        recon = recon.view(B, C, GH, GW, p, p).permute(0,1,2,4,3,5).contiguous().view(B, C, H2, W2)
        if pad_h or pad_w:
            recon = recon[:, :, :H, :W]
        return renormalize(recon.clamp(0, 1))

In [ ]:
_compressai_cache: Dict[tuple, nn.Module] = {}

def _get_compressai_model(level: int, dev: torch.device) -> nn.Module:
    key = (level, str(dev))
    if key not in _compressai_cache:
        net = cheng2020_attn(quality=level, pretrained=True).to(dev).eval()
        net.update()
        _compressai_cache[key] = net
    return _compressai_cache[key]

def _map_quality_to_lic_level(q: float) -> int:
    mapping = {20: 2, 80: 5}
    if q in mapping:
        return mapping[q]
    return max(1, min(6, int(round(1 + (q - 1) * 5 / 99))))


class LicRoiPerturbation(Perturbation):
    def __init__(self, quality: float):
        super().__init__(f"lic_roi_q{int(quality)}")
        self.quality   = quality
        self.base_level = _map_quality_to_lic_level(quality)
        self.hq_level   = min(6, self.base_level + 1)
        self.lq_level   = max(1, self.base_level - 1)

    def _compress(self, x_px, level, dev):
        net = _get_compressai_model(level, dev)
        with torch.no_grad():
            return net(x_px)["x_hat"].clamp(0, 1)

    def apply(self, model, images, labels, device):
        images = images.detach().to(device)
        x_px   = denormalize(images)
        _, _, H, W = x_px.shape
        ph = (64 - H % 64) % 64
        pw = (64 - W % 64) % 64
        if ph or pw:
            x_px = F.pad(x_px, (0, pw, 0, ph), mode="reflect")

        hq = self._compress(x_px, self.hq_level, device)
        lq = self._compress(x_px, self.lq_level, device)
        if ph or pw:
            hq = hq[:, :, :H, :W]
            lq = lq[:, :, :H, :W]

        if model is not None:
            # Gradient saliency mask
            xr = images.detach().requires_grad_(True)
            logits = model(xr)
            preds  = logits.argmax(1)
            logits[torch.arange(len(preds), device=device), preds].sum().backward()
            sal = xr.grad.abs().mean(1, keepdim=True)
            B   = sal.size(0)
            sal = (sal - sal.view(B,-1).min(1).values.view(B,1,1,1)) /                   (sal.view(B,-1).max(1).values.view(B,1,1,1) - sal.view(B,-1).min(1).values.view(B,1,1,1) + 1e-8)
            if sal.shape[2:] != hq.shape[2:]:
                sal = F.interpolate(sal, size=hq.shape[2:], mode="bilinear", align_corners=False)
            blended = sal * hq + (1 - sal) * lq
        else:
            blended = hq

        return renormalize(blended)

In [ ]:
class FgsmPerturbation(Perturbation):
    def __init__(self, epsilon: float):
        super().__init__(f"fgsm_eps{epsilon:.4f}")
        self.epsilon = float(epsilon)

    def apply(self, model, images, labels, device):
        x0 = denormalize(images.detach().to(device))
        x  = x0.clone().requires_grad_(True)
        loss = F.cross_entropy(model(renormalize(x)), labels.to(device))
        grad = torch.autograd.grad(loss, x)[0]
        return renormalize(project_linf_pixel(x0 + self.epsilon * grad.sign(), x0, self.epsilon))


class PGDPerturbation(Perturbation):
    def __init__(self, epsilon: float, steps: int = 10, alpha: float = None, random_start: bool = True):
        self.epsilon = float(epsilon)
        self.steps   = int(steps)
        self.alpha   = float(alpha) if alpha is not None else 2.0 / 255.0
        self.rs      = bool(random_start)
        super().__init__(f"pgd_eps{self.epsilon:.4f}_k{self.steps}")

    def apply(self, model, images, labels, device):
        y   = labels.to(device)
        x0  = denormalize(images.detach().to(device))
        xadv = project_linf_pixel(x0 + torch.empty_like(x0).uniform_(-self.epsilon, self.epsilon), x0, self.epsilon) if self.rs else x0.clone()
        for _ in range(self.steps):
            xadv = xadv.detach().requires_grad_(True)
            grad = torch.autograd.grad(F.cross_entropy(model(renormalize(xadv)), y), xadv)[0]
            with torch.no_grad():
                xadv = project_linf_pixel(xadv + self.alpha * grad.sign(), x0, self.epsilon)
        return renormalize(xadv.detach())


def dlr_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    s, idx = torch.sort(logits, dim=1, descending=True)
    zy   = logits[torch.arange(len(labels), device=logits.device), labels]
    zoth = torch.where(idx[:, 0] == labels, s[:, 1], s[:, 0])
    return -(zy - zoth) / (s[:, 0] - s[:, 2] + 1e-12)


class APGDPerturbation(Perturbation):
    def __init__(self, epsilon: float = 8/255, num_steps: int = 20, num_restarts: int = 1, loss_type: str = "dlr"):
        super().__init__(f"apgd_{loss_type}_eps{epsilon:.4f}")
        self.epsilon      = float(epsilon)
        self.num_steps    = int(num_steps)
        self.num_restarts = int(num_restarts)
        self.loss_type    = loss_type

    def apply(self, model, images, labels, device):
        y  = labels.to(device)
        x0 = denormalize(images.detach().to(device))
        xbest     = x0.clone()
        loss_best = torch.full((x0.size(0),), -1e10, device=device)
        c1, c2    = max(1, int(0.22*self.num_steps)), max(1, int(0.75*self.num_steps))
        for _ in range(self.num_restarts):
            x   = project_linf_pixel(x0 + torch.empty_like(x0).uniform_(-self.epsilon, self.epsilon), x0, self.epsilon)
            eta = 2.0 * self.epsilon
            for i in range(self.num_steps):
                x    = x.detach().requires_grad_(True)
                logits = model(renormalize(x))
                loss_i = dlr_loss(logits, y) if self.loss_type == "dlr" else F.cross_entropy(logits, y, reduction="none")
                grad   = torch.autograd.grad(loss_i.sum(), x)[0]
                with torch.no_grad():
                    improved = loss_i > loss_best
                    loss_best[improved] = loss_i[improved]
                    xbest[improved]     = x[improved]
                    x = project_linf_pixel(x + eta * grad.sign(), x0, self.epsilon)
                    if (i+1) in {c1, c2}: eta *= 0.5
        return renormalize(xbest.detach())

In [ ]:
compression_registry: Dict[str, Callable[[float], Perturbation]] = {
    "jpeg":     lambda q: JpegPerturbation(quality=int(q)),
    "jpeg2000": lambda q: Jpeg2000Perturbation(quality=float(q)),
    "pca":      lambda q: PcaPerturbation(quality=float(q)),
    "patchsvd": lambda q: PatchSVDPerturbation(quality=float(q), patch_size=8),
    "lic_roi":  lambda q: LicRoiPerturbation(quality=float(q)),
}

attack_registry: Dict[str, Callable[[float], Perturbation]] = {
    "fgsm": lambda eps: FgsmPerturbation(epsilon=eps),
    "pgd":  lambda eps: PGDPerturbation(epsilon=eps),
    "apgd": lambda eps: APGDPerturbation(epsilon=eps),
}

print("Registries ready.")

In [ ]:
def predict(img: torch.Tensor) -> Tuple[int, str, float]:
    """Return (class_idx, class_name, confidence) for a single normalised image tensor [1,C,H,W]."""
    with torch.no_grad():
        probs = torch.softmax(model(img), dim=1)
    conf, idx = probs.max(dim=1)
    return idx.item(), CLASS_NAMES[idx.item()], conf.item()

def image_psnr(img_a: torch.Tensor, img_b: torch.Tensor) -> float:
    """PSNR between two normalised image tensors (dB)."""
    mse = ((denormalize(img_a) - denormalize(img_b)) ** 2).mean().item()
    return float("inf") if mse < 1e-10 else 10 * math.log10(1.0 / mse)

def to_display(img: torch.Tensor, upscale: int = 1) -> np.ndarray:
    """Normalised tensor [1,C,H,W] → numpy [H,W,3] in [0,1], optionally upscaled (nearest-neighbour)."""
    px = denormalize(img).squeeze(0).cpu().permute(1, 2, 0).numpy().clip(0, 1)
    if upscale > 1:
        from PIL import Image as PILImage
        pil = PILImage.fromarray((px * 255).astype("uint8"))
        pil = pil.resize((px.shape[1] * upscale, px.shape[0] * upscale), PILImage.NEAREST)
        px  = np.asarray(pil) / 255.0
    return px

## Demo — Five Pipeline Variants
Runs the selected image through all five pipeline types and collects predictions.

In [ ]:
comp   = compression_registry[COMPRESSION_KEY](QUALITY)
attack = attack_registry[ATTACK_KEY](EPSILON)

def run_pipeline(steps: List[Perturbation]) -> torch.Tensor:
    x = img_norm
    for step in steps:
        x = step.apply(model, x, true_label_t, device)
    return x

PIPELINES: List[Tuple[str, torch.Tensor]] = [
    ("Clean",                        img_norm),
    (f"Compressed\n{COMPRESSION_KEY.upper()} Q={QUALITY}", run_pipeline([comp])),
    (f"Attacked\n{ATTACK_KEY.upper()} ε=8/255",            run_pipeline([attack])),
    (f"Comp → Attack\n{COMPRESSION_KEY.upper()} Q={QUALITY} → {ATTACK_KEY.upper()}", run_pipeline([comp, attack])),
    (f"Attack → Comp\n{ATTACK_KEY.upper()} → {COMPRESSION_KEY.upper()} Q={QUALITY}", run_pipeline([attack, comp])),
]

results = []
for name, img in PIPELINES:
    idx, cls, conf = predict(img)
    p = image_psnr(img_norm, img) if name != "Clean" else float("inf")
    correct = (idx == true_label)
    results.append({"name": name, "img": img, "class": cls, "conf": conf, "psnr": p, "correct": correct})
    tick = "✓" if correct else "✗"
    psnr_str = f"{p:5.1f} dB" if p != float("inf") else "    ∞"
    print(f"{name.replace(chr(10),' '):<42}  {tick}  {cls:<22}  {conf:.1%}   PSNR {psnr_str}")

In [ ]:
UPSCALE = 7 if DATASET_NAME == "cifar10" else 1   # blow up 32x32 for visibility
BG      = "#12121f"
CORRECT_COL = "#00e676"
WRONG_COL   = "#ff1744"
PSNR_COL    = "#ffd740"

fig, axes = plt.subplots(1, len(results), figsize=(4.2 * len(results), 5.5))
fig.patch.set_facecolor(BG)

for ax, r in zip(axes, results):
    img_np = to_display(r["img"], upscale=UPSCALE)
    ax.imshow(img_np, interpolation="nearest", aspect="equal")
    ax.set_xticks([]); ax.set_yticks([])

    ec = CORRECT_COL if r["correct"] else WRONG_COL
    for sp in ax.spines.values():
        sp.set_edgecolor(ec)
        sp.set_linewidth(3.5)
    ax.set_facecolor(BG)

    # Pipeline name as title
    ax.set_title(r["name"], color="white", fontsize=11, fontweight="bold",
                 pad=10, linespacing=1.4)

    # Prediction + confidence as x-label
    ax.set_xlabel(f"{r['class']}\n{r['conf']:.1%}", color=ec, fontsize=10.5,
                  fontweight="bold", labelpad=8, linespacing=1.5)

    # PSNR badge (skip for Clean)
    if r["psnr"] != float("inf"):
        ax.text(0.97, 0.03, f"PSNR\n{r['psnr']:.1f} dB",
                transform=ax.transAxes, color=PSNR_COL, fontsize=8,
                ha="right", va="bottom", linespacing=1.3,
                bbox=dict(boxstyle="round,pad=0.35", facecolor="black", alpha=0.65, edgecolor="none"))

# True-class header
fig.suptitle(
    f"True class: {true_class}   ·   Model: {MODEL_NAME}   ·   Image #{IMAGE_INDEX}",
    color="white", fontsize=13, fontweight="bold", y=1.03,
)

plt.tight_layout(pad=1.5)

if FIGURE_SAVE_PATH:
    plt.savefig(FIGURE_SAVE_PATH, dpi=150, bbox_inches="tight", facecolor=BG)
    print(f"Saved → {FIGURE_SAVE_PATH}")

plt.show()

## Multi-Quality Comparison

Shows the same image across all three quality levels (25 / 50 / 75) for one pipeline type.  
Set `MULTI_PIPELINE` to `"comp_only"`, `"attack_comp"`, or `"comp_attack"`.

In [ ]:
MULTI_PIPELINE = "attack_comp"   # "comp_only" | "comp_attack" | "attack_comp"
QUALITIES      = [25, 50, 75]

def run_multi_quality(pipeline_type: str, quality: float) -> torch.Tensor:
    c = compression_registry[COMPRESSION_KEY](quality)
    a = attack_registry[ATTACK_KEY](EPSILON)
    if pipeline_type == "comp_only":
        return run_pipeline([c])
    elif pipeline_type == "comp_attack":
        return run_pipeline([c, a])
    elif pipeline_type == "attack_comp":
        return run_pipeline([a, c])
    else:
        raise ValueError(pipeline_type)

mq_results = []
for q in QUALITIES:
    img_out = run_multi_quality(MULTI_PIPELINE, q)
    idx, cls, conf = predict(img_out)
    p = image_psnr(img_norm, img_out)
    mq_results.append({"q": q, "img": img_out, "class": cls, "conf": conf,
                        "psnr": p, "correct": idx == true_label})
    tick = "✓" if idx == true_label else "✗"
    print(f"Q={q:<3}  {tick}  {cls:<22}  {conf:.1%}   PSNR {p:.1f} dB")

In [ ]:
n_panels = 1 + len(mq_results)   # Clean + one per quality
fig, axes = plt.subplots(1, n_panels, figsize=(4.2 * n_panels, 5.5))
fig.patch.set_facecolor(BG)

all_panels = [{"name": "Clean", "img": img_norm, "class": CLASS_NAMES[true_label],
               "conf": predict(img_norm)[2], "psnr": float("inf"),
               "correct": True}] +              [{"name": f"{COMPRESSION_KEY.upper()} Q={r['q']}", **r} for r in mq_results]

for ax, r in zip(axes, all_panels):
    img_np = to_display(r["img"], upscale=UPSCALE)
    ax.imshow(img_np, interpolation="nearest", aspect="equal")
    ax.set_xticks([]); ax.set_yticks([])

    ec = CORRECT_COL if r["correct"] else WRONG_COL
    for sp in ax.spines.values():
        sp.set_edgecolor(ec); sp.set_linewidth(3.5)
    ax.set_facecolor(BG)

    ax.set_title(r["name"], color="white", fontsize=11, fontweight="bold", pad=10)
    ax.set_xlabel(f"{r['class']}\n{r['conf']:.1%}", color=ec,
                  fontsize=10.5, fontweight="bold", labelpad=8, linespacing=1.5)

    if r["psnr"] != float("inf"):
        ax.text(0.97, 0.03, f"PSNR\n{r['psnr']:.1f} dB",
                transform=ax.transAxes, color=PSNR_COL, fontsize=8,
                ha="right", va="bottom", linespacing=1.3,
                bbox=dict(boxstyle="round,pad=0.35", facecolor="black", alpha=0.65, edgecolor="none"))

pipeline_label = {"comp_only": "Compression only",
                  "comp_attack": "Compress → Attack",
                  "attack_comp": "Attack → Compress"}[MULTI_PIPELINE]
fig.suptitle(
    f"True: {true_class}   ·   {pipeline_label}   ·   {COMPRESSION_KEY.upper()} at Q=25/50/75   ·   {ATTACK_KEY.upper()} ε=8/255",
    color="white", fontsize=12, fontweight="bold", y=1.03,
)

plt.tight_layout(pad=1.5)
save_path = f"demo_{DATASET_NAME}_{COMPRESSION_KEY}_multiq_{MULTI_PIPELINE}.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=BG)
print(f"Saved → {save_path}")
plt.show()